# Environmental VOI Tutorial

This notebook turns a small environmental-policy example into a VOI problem.
It combines simple impact accounting with a decision analysis over three policy
options: hold, target, or retrofit.

In [1]:
import numpy as np

from voiage.analysis import DecisionAnalysis
from voiage.environmental.impact_assessment import (
    calculate_carbon_footprint,
    calculate_water_usage,
    environmental_lifecycle_assessment,
)
from voiage.schema import ValueArray

activities = {
    "electricity_kwh": 1200.0,
    "fuel_liters": 180.0,
}
emission_factors = {
    "electricity_kwh": 0.42,
    "fuel_liters": 2.31,
}
water_processes = {
    "cooling": 120.0,
    "cleaning": 35.0,
}
water_factors = {
    "cooling": 14.0,
    "cleaning": 8.0,
}
stage_impacts = {
    "production": {"carbon": 120.0, "water": 35.0},
    "operation": {"carbon": 85.0, "water": 16.0},
    "end_of_life": {"carbon": 20.0, "water": 4.0},
}
valuation_factors = {"carbon": 85.0, "water": 0.6}

carbon_footprint = calculate_carbon_footprint(activities, emission_factors)
water_usage = calculate_water_usage(water_processes, water_factors)
lifecycle = environmental_lifecycle_assessment(
    ["production", "operation", "end_of_life"],
    stage_impacts,
    valuation_factors,
)

print(f"Carbon footprint: {carbon_footprint:.2f} kg CO2e")
print(f"Water usage: {water_usage:.2f} liters")
print(f"Lifecycle total cost: {lifecycle['total_cost']:.2f}")
print("Lifecycle stage costs:", lifecycle["stage_costs"])

Carbon footprint: 919.80 kg CO2e
Water usage: 1960.00 liters
Lifecycle total cost: 19158.00
Lifecycle stage costs: {'production': {'carbon': 10200.0, 'water': 21.0}, 'operation': {'carbon': 7225.0, 'water': 9.6}, 'end_of_life': {'carbon': 1700.0, 'water': 2.4}}


In [2]:
rng = np.random.default_rng(7)
n_samples = 500
abatement_efficiency = rng.beta(3.0, 2.0, size=n_samples)
compliance_rate = rng.beta(4.0, 2.0, size=n_samples)
carbon_price = rng.normal(85.0, 10.0, size=n_samples)
implementation_cost = np.clip(rng.normal(18.0, 4.0, size=n_samples), 6.0, None)

net_benefits = np.column_stack([
    np.zeros(n_samples),
    10.0 + 25.0 * abatement_efficiency * compliance_rate + 0.08 * carbon_price - implementation_cost,
    12.0 + 35.0 * abatement_efficiency * compliance_rate + 0.12 * carbon_price - 1.25 * implementation_cost,
])

strategy_names = ["Hold", "Targeted controls", "Full retrofit"]
analysis = DecisionAnalysis(
    nb_array=ValueArray.from_numpy(net_benefits, strategy_names),
    parameter_samples={
        "abatement_efficiency": abatement_efficiency,
        "compliance_rate": compliance_rate,
        "carbon_price": carbon_price,
    },
)

evpi = analysis.evpi()
evppi_carbon = analysis.evppi(["carbon_price"])
print(f"EVPI: {evpi:.4f}")
print(f"EVPPI for carbon price: {evppi_carbon:.4f}")
print("Expected net benefits:", dict(zip(strategy_names, np.round(net_benefits.mean(axis=0), 4), strict=True)))

EVPI: 0.1185
EVPPI for carbon price: 0.0000
Expected net benefits: {'Hold': np.float64(0.0), 'Targeted controls': np.float64(8.9464), 'Full retrofit': np.float64(13.8948)}
